In [ ]:
import os
device = "cuda"
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('../..')

from DIGNN.data import AtomsData, ase2AtomsData
from DIGNN.utils import AtomIndexMapper, plot_comparison
from DIGNN.pl import DataModule, TrainModule_FF, TrainModule
from DIGNN.nn import models as dgm

import time
import random
import torch
import numpy as np
import pytorch_lightning as pl
from ase.build import molecule

In [ ]:
hyperparams = {"pml_rcut": 2.0, "pml_mnn": 12, "iml_rcut": 4.0, "iml_mnn": 16}
DIGNN_feat_dim = {'atom_dim': 64, 'bond_dim': 64, 'ang_dim': 32, 'dih_dim': 16}

In [ ]:
H2O2 = molecule('H2O2')
H2O2.arrays['energy'] = np.array([0.0])
H2O2.arrays['force'] = np.zeros_like(H2O2.positions)
H2O2.arrays['formation'] = np.array([1.0])
atomsdata = [ase2AtomsData(H2O2, check_rcut=hyperparams["pml_rcut"], properties=['energy', 'force', 'formation']) for _ in range(100)]

CH4 = molecule('CH4')
CH4.arrays['energy'] = np.array([0.0])
CH4.arrays['force'] = np.zeros_like(CH4.positions)
CH4.arrays['formation'] = np.array([1.0])
atomsdata += [ase2AtomsData(CH4, check_rcut=hyperparams["pml_rcut"], properties=['energy', 'force', 'formation']) for _ in range(100)]

C6H6 = molecule('C6H6')
C6H6.arrays['energy'] = np.array([0.0])
C6H6.arrays['force'] = np.zeros_like(C6H6.positions)
C6H6.arrays['formation'] = np.array([1.0])
atomsdata += [ase2AtomsData(C6H6, check_rcut=hyperparams["pml_rcut"], properties=['energy', 'force', 'formation']) for _ in range(100)]

random.shuffle(atomsdata)

## 性质训练测试

In [ ]:
data = DataModule(atomsdata, 
                    **hyperparams,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='cplt',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

In [ ]:
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **DIGNN_feat_dim,
                                            pml_rcut=hyperparams["pml_rcut"]+0.2,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            iml_rcut=hyperparams["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**DIGNN_feat_dim,
                                            pml=1,
                                            iml=4,
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            init_nn_layer=0,
                                            ), 
                decoder=dgm.Decoder(dim=[DIGNN_feat_dim['atom_dim'],64,1], 
                                    reduce_method='mean', 
                                    dropout=0.0),
                global_processor=dgm.modules.processor_components.Global_node_nn(
                                    atom_dim=DIGNN_feat_dim['atom_dim'],
                                    bond_dim=DIGNN_feat_dim['bond_dim'],
                                    global_node_num=6,
                                    proj_hid_dim=[128,128],
                                    layer_num=4, #iml
                                    residual=True,
                                    )
                ).to(device)
model.processor.lcp.global_processor.cuda()

In [ ]:
max_epoch = 5

train_module = TrainModule(model, 
                           compile_model=False, # 开启编译, 速度更快
                           lr=1e-3,
                           prop='formation',
                           adamw_weight_decay=1e-2,
                           adamw_betas=(0.9, 0.999),
                           onecycle_total_steps=max_epoch*len(data.train_dataloader()), 
                           onecycle_final_div_factor=1e+5,
                           empty_cache_every_epoch=True,
                           )
trainer = pl.Trainer(max_epochs=max_epoch,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=1,
                    precision='16-mixed',
                    benchmark=True,
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

preds, targets = train_module.test_results.values()
plot_comparison(target=targets, pred=preds)

In [ ]:
train_module.test_results

## 力场训练测试

In [ ]:
data = DataModule(atomsdata, 
                    **hyperparams,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='basic',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup


In [ ]:
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **DIGNN_feat_dim,
                                            pml_rcut=hyperparams["pml_rcut"]+0.2,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            iml_rcut=hyperparams["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**DIGNN_feat_dim,
                                            pml=1,
                                            iml=4,
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=DIGNN_feat_dim['bond_dim'],
                                            init_nn_layer=0,
                                            ), 
                decoder=dgm.Decoder(dim=[DIGNN_feat_dim['atom_dim'],64,1], 
                                    reduce_method='sum', 
                                    dropout=0.0),
                global_processor=dgm.modules.processor_components.Global_node_nn(
                                    atom_dim=DIGNN_feat_dim['atom_dim'],
                                    bond_dim=DIGNN_feat_dim['bond_dim'],
                                    global_node_num=6,
                                    proj_hid_dim=[128,128],
                                    layer_num=4, #iml
                                    residual=True,
                                    )
                ).to(device)
model.processor.lcp.global_processor.cuda()

In [ ]:
max_epoch = 5

train_module = TrainModule_FF(model,
                              compile_model=False, # FF训练不开启
                              lr=1e-3,
                              energy_weight=0.1,
                              force_weight=1.0,
                              adamw_weight_decay=1e-2,
                              adamw_betas=(0.9, 0.999),
                              onecycle_total_steps=max_epoch*len(data.train_dataloader()),
                              onecycle_final_div_factor=1e+5,
                              )
trainer = pl.Trainer(max_epochs=max_epoch,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=50,
                    benchmark=True,
                    inference_mode=False,
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)


In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

preds_ene, targets_ene, preds_force, targets_force, atom_num = train_module.test_results.values()
plot_comparison(target=targets_ene, pred=preds_ene, atom_num=atom_num)
# plot_comparison(target=targets_force, pred=preds_force)